# Chapter 08 — GPU Profiling with PyTorch

Deep learning runs on GPUs — this is not optional, it's a fundamental requirement. A single matrix multiplication that takes minutes on a CPU completes in milliseconds on a GPU. This chapter explains why GPUs are so much faster, how CUDA programming works, and how to write efficient code that fully utilizes your hardware.

### Glossary / Glossário

• CUDA kernel — function that runs on the GPU across many parallel threads. / Função que executa na GPU em muitas threads paralelas.
• VRAM — Video RAM, the GPU's dedicated memory for storing tensors and model weights. / Video RAM, memória dedicada da GPU para armazenar tensores e pesos do modelo.
• shared memory — fast on-chip SRAM shared by threads within a GPU thread block. / SRAM rápida on-chip compartilhada por threads dentro de um bloco GPU.
• profiler — tool that measures execution time and resource usage of GPU operations. / Ferramenta que mede tempo de execução e uso de recursos de operações GPU.
• GPU-Util — GPU utilization percentage, how much of the time the GPU is actively computing. / Percentual de utilização da GPU, quanto do tempo ela está ativamente computando.

In [ ]:
import torch
import torch.nn as nn
from torch.profiler import profile, ProfilerActivity

# === Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # compute device — GPU if available, else CPU
print(f"Using: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")  # total GPU memory in gigabytes

# Simple model for profiling demo
model = nn.Sequential(
    nn.Linear(768, 3072),   # FFN up-projection (4x expansion)
    nn.GELU(),              # activation function
    nn.Linear(3072, 768),   # FFN down-projection (back to model dim)
).to(device)

x = torch.randn(32, 512, 768, device=device)  # batch=32, seq_len=512, hidden=768

# === Aha: Profile naive vs optimized ===
# Naive: sync after each forward pass
torch.cuda.synchronize() if device.type == "cuda" else None
import time

start = time.perf_counter()
for _ in range(10):
    output = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
naive_time = (time.perf_counter() - start) / 10

# Optimized: no unnecessary syncs
start = time.perf_counter()
for _ in range(10):
    output = model(x)
if device.type == "cuda":
    torch.cuda.synchronize()
opt_time = (time.perf_counter() - start) / 10

print(f"\n=== Before vs After Optimization ===")
print(f"Naive (sync each iter):   {naive_time*1000:.2f} ms/iter")
print(f"Optimized (sync at end):  {opt_time*1000:.2f} ms/iter")
print(f"Speedup: {naive_time/opt_time:.2f}x")
print(f"\nKey insight: unnecessary torch.cuda.synchronize() calls kill performance!")

---

**Course:** Aprenda LLM | **Chapter 08**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.